# Chapter 11 — Deeper and southern: Reticulum II and the tails of Tucana III in DES

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/11_des_satellites.ipynb)

**Goal.** The Dark Energy Survey (DES) imaged 5000 deg² of the southern sky about 1.5–2 magnitudes deeper than SDSS.
In its first year it revealed more than a dozen new satellites (Bechtol et al. 2015, https://arxiv.org/abs/1503.02584;
Koposov et al. 2015, https://doi.org/10.1088/0004-637X/805/2/130). We apply the methods of Chapters 2, 5 and 6 to two of
them: **Reticulum II**, a compact ultra-faint dwarf, and **Tucana III**, a satellite being torn apart by the Milky Way,
with tidal tails on both sides (Drlica-Wagner et al. 2015; Shipp et al. 2018, https://ui.adsabs.harvard.edu/abs/2018ApJ...862..114S).

**You will learn**
1. to adapt the pipeline to another survey: its magnitudes, star–galaxy flag, extinction coefficients and isochrone system;
2. that a deeper survey reaches the main-sequence turn-off of satellites at 30–60 kpc, multiplying the number of members;
3. to see a satellite and its tails in one map, and measure the tails' direction and significance.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter        # smoothing of 2-d histograms (CMDs and sky maps)
from scipy.optimize import minimize              # maximum-likelihood fit of the structure (Chapter 6)
from scipy import stats                          # Poisson and Gaussian tail probabilities

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')             # Local Volume Database: catalogued positions, distances, structure
iso = load('isochrones_des')                     # PARSEC isochrones in the DES (DECam) g, r, i bands

## 11.1 DES photometry

The files hold DES DR2 point sources (`extended_class_coadd` 0 or 1: high-confidence and probable stars) with
weighted-average PSF magnitudes. DR2 magnitudes are not dereddened; the extinction is $A_b = R_b\,E(B-V)_{\rm SFD98}$ with
$R_g = 3.186$, $R_r = 2.140$, $R_i = 1.569$ (Abbott et al. 2021, https://ui.adsabs.harvard.edu/abs/2021ApJS..255...20A,
Sect. 4.2). The isochrones are PARSEC in the DES (DECam) bands.

In [ ]:
R_EXT = {'g': 3.186, 'r': 2.140, 'i': 1.569}     # A_b / E(B-V)_SFD98 for DES DR2 (Abbott et al. 2021, Sect. 4.2)


def dered(d):
    """Extinction-corrected g0, r0 [mag] from the DES DR2 weighted-average PSF magnitudes and the SFD98 E(B-V)."""
    g = (d.wavg_mag_psf_g - R_EXT['g'] * d.ebv_sfd98).to_numpy()
    r = (d.wavg_mag_psf_r - R_EXT['r'] * d.ebv_sfd98).to_numpy()
    return g, r


# CMD grid: colour (g-r)_0 from -0.4 to 1.0 in 0.04 mag bins (35), magnitude g_0 from 16 to 24 in 0.1 mag bins (80)
cb, mb = np.arange(-0.4, 1.01, 0.04), np.arange(16, 24.01, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])          # bin centres
CC, MM = np.meshgrid(cc, mm, indexing='ij')                          # shape (35, 80): [colour, magnitude]


def satellite_cmd(mu, logage=10.1, feh=-2.1, err=lambda m: 0.01 + 0.08 * np.exp((m - 23.5) / 0.9)):
    """Normalised CMD density of an old population in DES bands (as in Chapters 4-5; a DES-like error law).

    mu: distance modulus; logage = 10.1 (12.6 Gyr) and [Fe/H] = -2.1: an old, metal-poor population. err(m): magnitude
    error [mag] at magnitude m (0.09 mag at g = 23.5, growing by a factor e every 0.9 mag). Returns an array of shape
    (35, 80) on the (cc, mm) grid that sums to 1.
    """
    # PARSEC evolutionary stages 0-7 (up to the early AGB), ordered by initial mass, i.e. along the sequence
    I = iso[np.isclose(iso.logage, logage) & np.isclose(iso.feh, feh) & (iso.stage <= 7)].sort_values('m_ini')
    c, g = (I.g - I.r).to_numpy(), I.g.to_numpy()
    # resample the isochrone at 3000 points equally spaced in CMD arc length, so that phases where the track moves
    # fast (subgiant branch, giant branch) are not under-sampled
    arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(c), np.diff(g)))])
    s = np.linspace(0, arc[-1], 3000)
    # point colour, apparent magnitude, and weight = fraction of stars (from the cumulative IMF) at each point
    pc, pg, w = np.interp(s, arc, c), np.interp(s, arc, g) + mu, np.gradient(np.interp(s, arc, I.int_imf))
    # Gaussian widths in colour (two bands: sqrt(2) x error) and magnitude, with floors of 0.03 and 0.1 mag added in
    # quadrature, so that the sequence is never narrower than about one CMD bin
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    # sum over the 3000 isochrone points k of their weighted 2-d Gaussians on the grid -> shape (35, 80)
    S = np.einsum('k,kij->ij', w, np.exp(-0.5 * ((CC - pc[:, None, None]) / sc[:, None, None])**2
                                         - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_mask(S, B, keep=0.5):
    """Boolean CMD mask (Chapter 4): cells ranked by signal-to-background S/B, cut where signal/sqrt(background) peaks,
    among masks holding at least a fraction `keep` of the satellite's stars."""
    order = np.argsort((S / B).ravel())[::-1]                    # cells from the highest to the lowest S/B
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])   # cumulative signal and background fractions
    M = np.zeros(S.size, bool)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


def in_mask(c, m, M):
    """True for stars (colour c, magnitude m) that fall in a cell of mask M; stars outside the grid are False."""
    i, j = np.digitize(c, cb) - 1, np.digitize(m, mb) - 1        # colour and magnitude bin indices
    ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
    return ok & M[np.clip(i, 0, len(cc) - 1), np.clip(j, 0, len(mm) - 1)]

## 11.2 Reticulum II

Reticulum II lies at $\mu = 17.5$ (32 kpc), close enough that DES reaches well below its main-sequence turn-off
($g_0 \approx 21.5$). The on − off CMD shows a sequence of ~1000 stars (the excess printed below) from the giant branch to the faint main sequence.

In [ ]:
ref = lvdb.loc['reticulum_2']
cat = load('retii_des')                          # DES DR2 point sources around Reticulum II
g0, r0 = dered(cat)
c0 = g0 - r0
x, y = tangent(cat.ra.to_numpy(), cat.dec.to_numpy(), ref.ra, ref.dec)    # [deg]
R = np.hypot(x, y) * 60                          # distance from the catalogued centre [arcmin]
# on: within 1.5 r_h (9.5' for r_h = 6.3'); off: a background annulus at 60'-110', far outside the satellite
on, off = R < 1.5 * ref.rhalf, (R > 60) & (R < 110)
alpha = (1.5 * ref.rhalf)**2 / (110**2 - 60**2)  # area ratio on/off: scales the off counts to the on area
# the same isochrone as in satellite_cmd: 12.6 Gyr, [Fe/H] = -2.1
I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')

# Left: CMD of on (red) and off (grey) stars with the isochrone at the catalogued distance modulus (blue)
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
ax[0].scatter(c0[off], g0[off], s=0.3, c='0.7', label='60′–110′ (all)')
ax[0].scatter(c0[on], g0[on], s=5, c='C3', label=f'R < {1.5 * ref.rhalf:.1f}′')
ax[0].plot(I.g - I.r, I.g + ref.distance_modulus, c='C0', lw=1, label=f'PARSEC 12.6 Gyr, [Fe/H] = −2.1, μ = {ref.distance_modulus:.1f}')
ax[0].set(xlim=(-0.4, 1.0), ylim=(24, 16), xlabel='$(g-r)_0$', ylabel='$g_0$', title='Reticulum II in DES DR2')
ax[0].legend(fontsize=7, markerscale=3)
# Right: background-subtracted Hess diagram, on - alpha * off, in coarser bins (every 2nd colour edge: 0.08 mag; every
# 3rd magnitude edge: 0.3 mag) so that each bin holds a few stars; red = excess, blue = deficit, scale +-5 stars per bin
dH = np.histogram2d(c0[on], g0[on], [cb[::2], mb[::3]])[0] - alpha * np.histogram2d(c0[off], g0[off], [cb[::2], mb[::3]])[0]
ax[1].imshow(dH.T, origin='upper', extent=[cb[0], cb[-1], mb[-1], mb[0]], aspect='auto', cmap='RdBu_r', vmin=-5, vmax=5)
ax[1].plot(I.g - I.r, I.g + ref.distance_modulus, c='k', lw=0.8)
ax[1].set(xlim=(-0.4, 1.0), xlabel='$(g-r)_0$', title='on − α off')
plt.tight_layout()
# excess = N_on - alpha N_off, with its Poisson error sqrt(N_on + alpha^2 N_off)
print(f'on {on.sum()}, scaled background {alpha * off.sum():.0f}, excess {on.sum() - alpha * off.sum():.0f} ± {np.sqrt(on.sum() + alpha**2 * off.sum()):.0f}')

**Figure 11.1.** The CMD of Reticulum II in DES DR2 (point sources, `extended_class_coadd` ≤ 1; PSF magnitudes corrected
for SFD98 extinction). Left: $(g-r)_0$ against $g_0$ for the 2752 stars within $R < 1.5\,r_h = 9.5′$ of the centre (red)
and for all stars in the 60′–110′ background annulus (grey), with a PARSEC isochrone of 12.6 Gyr and [Fe/H] = −2.1
shifted to the catalogued μ = 17.5 (blue line). Right: the background-subtracted Hess diagram, on − α·off (α = ratio of
the on and off areas), in bins of 0.08 mag in colour and 0.3 mag in $g_0$; red is an excess and blue a deficit, on a scale
of ±5 stars per bin, with the same isochrone in black. The excess within 9.5′ is 965 ± 53 stars, concentrated along the
turn-off and main sequence at $g_0 \approx 21$–24 and $(g-r)_0 \approx 0.2$–0.45. Compare with Boötes I in SDSS
(Chapter 2): there we saw only the giant branch and horizontal branch of a satellite at 66 kpc; here the turn-off and the
upper main sequence of a satellite at 32 kpc dominate the excess.

### Structure, with the likelihood of Chapter 6

We select stars in the CMD mask at the catalogued distance and fit an elliptical exponential plus a uniform background
within 1° (Chapter 6, same model and code).

In [ ]:
# Background CMD density: Hess diagram of all stars in the field (dominated by field stars), smoothed by one bin and
# floored at 1 count per cell so that S/B is finite everywhere; normalised to sum to 1
H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
# stars in the CMD mask for the catalogued distance modulus, within 60' of the centre
sel = in_mask(c0, g0, cmd_mask(satellite_cmd(ref.distance_modulus), B)) & (R < 60)
xs, ys = x[sel] * 60, y[sel] * 60                # tangent-plane coordinates [arcmin]
R60, RH_MIN = 60.0, 0.5                          # radius of the fitted region and lower bound on r_h [arcmin]
P_BG = 1 / (np.pi * R60**2)                      # uniform background: normalised density over the 60' disc [arcmin^-2]


def p_sat(x, y, x0, y0, rh, eps, theta):
    """Elliptical exponential profile (Chapter 6), normalised over the plane [arcmin^-2].

    Centre (x0, y0) and half-light radius rh (semi-major axis) in arcmin, ellipticity eps = 1 - b/a, position angle
    theta [deg] of the major axis, east of north.
    """
    a = rh / 1.678                               # exponential scale length: r_h = 1.678 a for a 2-d exponential
    t = np.radians(theta)
    u = (x - x0) * np.sin(t) + (y - y0) * np.cos(t)          # along the major axis
    v = (x - x0) * np.cos(t) - (y - y0) * np.sin(t)          # along the minor axis
    return np.exp(-np.sqrt(u**2 + (v / (1 - eps))**2) / a) / (2 * np.pi * a**2 * (1 - eps))


def unpack(q):
    """Unconstrained fit parameters q -> (x0, y0, r_h, eps, PA, f) with r_h > RH_MIN, 0 < eps < 0.95, 0 <= PA < 180,
    0 < f < 1 (f: fraction of the masked stars that belong to the satellite)."""
    return q[0], q[1], RH_MIN + np.exp(q[2]), 0.95 / (1 + np.exp(-q[3])), q[4] % 180, 1 / (1 + np.exp(-q[5]))


def nll(q, x, y):
    """Negative log-likelihood of the stars (x, y) [arcmin] under satellite + uniform background."""
    x0, y0, rh, eps, th, f = unpack(q)
    return -np.sum(np.log(f * p_sat(x, y, x0, y0, rh, eps, th) + (1 - f) * P_BG))


# Nelder-Mead from three starting position angles (0, 60, 120 deg), keeping the lowest -log L, to avoid stopping in a
# local minimum in PA. Start: centre at the catalogued position, r_h = 0.5 + 5', eps = 0.95 / (1 + e^-0.5) = 0.59,
# f = 1 / (1 + e^2) = 0.12.
best = min((minimize(nll, [0, 0, np.log(5), 0.5, th0, -2], args=(xs, ys), method='Nelder-Mead',
                     options=dict(maxiter=8000, xatol=1e-5, fatol=1e-6)) for th0 in (0, 60, 120)), key=lambda r: r.fun)
p = np.array(unpack(best.x))                     # best-fit (x0, y0, r_h, eps, PA, f)
# to_q: the inverse of unpack (physical -> unconstrained parameters)
to_q = lambda p: np.array([p[0], p[1], np.log(p[2] - RH_MIN), -np.log(0.95 / p[3] - 1), p[4], -np.log(1 / p[5] - 1)])
# Uncertainties: Hessian of -log L in the physical parameters by central finite differences; the covariance is its
# inverse. Steps: 0.05' in x0 and y0, 2% of r_h, 0.01 in eps, 1 deg in PA, 0.002 in f.
h = np.array([0.05, 0.05, 0.02 * p[2], 0.01, 1.0, 0.002])
Hm = np.zeros((6, 6))
for a_ in range(6):
    for b_ in range(6):
        ea, eb = np.eye(6)[a_] * h[a_], np.eye(6)[b_] * h[b_]
        f = lambda pp: nll(to_q(pp), xs, ys)
        Hm[a_, b_] = (f(p + ea + eb) - f(p + ea - eb) - f(p - ea + eb) + f(p - ea - eb)) / (4 * h[a_] * h[b_])
err = np.sqrt(np.diag(np.linalg.inv(Hm)))        # 1-sigma errors of the six parameters
names = ['x0 [′]', 'y0 [′]', 'r_h [′]', 'eps', 'PA [deg]', 'f']
for n, v, e in zip(names, p, err):
    print(f'{n:9s} = {v:7.2f} ± {e:.2f}')
print(f'N_star = {p[5] * len(xs):.0f} of {len(xs)} masked stars')
print(f'LVDB (Mutlu-Pakdil et al. 2018): r_h = {ref.rhalf:.1f} ± {ref.rhalf_em:.1f}′, eps = {ref.ellipticity:.2f} ± '
      f'{ref.ellipticity_em:.2f}, PA = {ref.position_angle:.0f} ± {ref.position_angle_em:.0f}°')
dpa = abs((p[4] - ref.position_angle + 90) % 180 - 90)    # PA difference, folded to 0-90 deg (PA is defined modulo 180)
# Checks against the LVDB: centre within 1' of the catalogued position; r_h within 2 sigma (our error and the LVDB error
# in quadrature); ellipticity within 0.15 (1.5 x the LVDB error of 0.10) and PA within 15 deg
assert np.hypot(p[0], p[1]) < 1.0
assert abs(p[2] - ref.rhalf) < 2 * np.hypot(err[2], ref.rhalf_em)
assert abs(p[3] - ref.ellipticity) < 0.15 and dpa < 15

Reticulum II is very elongated (ε ≈ 0.6), and the DES fit agrees with the deeper Magellan imaging used by the LVDB
(Mutlu-Pakdil et al. 2018, https://ui.adsabs.harvard.edu/abs/2018ApJ...863...25M) within the errors. With DES depth,
hundreds of masked stars constrain the shape well — compare the 400 stars of the much more luminous Boötes I in SDSS.

## 11.3 Tucana III and its tidal tails

Tucana III ($\mu = 16.8$, 23 kpc) is a faint satellite on an orbit that takes it close to the Galactic centre; tidal
forces have pulled it into a stream ~4° long. We map the masked stars over 4.5° around it.

In [ ]:
t3 = lvdb.loc['tucana_3']
cat3 = load('tuc3_des')                          # DES DR2 point sources within ~4.5 deg of Tucana III
g3, r3 = dered(cat3)
c3 = g3 - r3
x3, y3 = tangent(cat3.ra.to_numpy(), cat3.dec.to_numpy(), t3.ra, t3.dec)  # [deg]
# CMD mask as for Reticulum II, with the background CMD of this field and the catalogued distance modulus
H3 = gaussian_filter(np.histogram2d(c3, g3, [cb, mb])[0], 1.0)
B3 = np.maximum(H3, 1.0) / np.maximum(H3, 1.0).sum()
sel3 = in_mask(c3, g3, cmd_mask(satellite_cmd(t3.distance_modulus), B3))
xe = np.arange(-4.5, 4.51, 0.05)                 # sky-map pixel edges [deg]: 0.05 deg pixels, 180 x 180
# masked counts per pixel, smoothed with a Gaussian of 2 pixels = 0.1 deg
img = gaussian_filter(np.histogram2d(x3[sel3], y3[sel3], [xe, xe])[0], 2)
cover = gaussian_filter(np.histogram2d(x3, y3, [xe, xe])[0] > 0, 2)            # crude footprint: any star in the pixel
fig, ax = plt.subplots(figsize=(7, 6))
XC, YC = np.meshgrid(0.5 * (xe[1:] + xe[:-1]), 0.5 * (xe[1:] + xe[:-1]), indexing='ij')   # pixel centres [deg]
# Divide by the smoothed footprint to correct the smoothing near edges and holes; show only well-covered pixels
# (cover > 0.5) within 4.4 deg. Linear grey scale saturated at the 99.7th percentile so that the stream is visible.
ax.imshow(np.where((cover > 0.5) & (np.hypot(XC, YC) < 4.4), img / np.maximum(cover, 1e-3), np.nan).T, origin='lower', extent=[xe[0], xe[-1], xe[0], xe[-1]],
          cmap='Greys', vmax=np.nanpercentile(img, 99.7))
ax.invert_xaxis()                                # east to the left, as on the sky
ax.set(xlabel=r'$\xi$ [deg]', ylabel=r'$\eta$ [deg]', title='Tucana III: masked DES stars, smoothed 0.1°')
plt.tight_layout()

**Figure 11.2.** Matched-filter star counts around Tucana III: DES DR2 stars inside the CMD mask built for a 12.6 Gyr,
[Fe/H] = −2.1 population at the catalogued distance modulus, counted in 0.05° pixels in the tangent plane ($\xi$ towards
east, plotted increasing to the left; $\eta$ north), smoothed with a Gaussian of 0.1° and divided by the equally smoothed
footprint; only pixels within 4.4° of the satellite are shown. Linear grey scale, black at the 99.7th percentile of the
smoothed counts. The compact core sits at the centre, and a thin stream ~4.5° long (from $\xi \approx +2°$ to
$\xi \approx -2.7°$) runs through it roughly east–west, bending slightly to the south at its western end
($\eta \approx -0.3°$ at $\xi \approx -2.5°$).

### Direction and significance of the tails

As for Pal 5 (Chapter 7), we choose the orientation of a line through the satellite that maximises the masked counts in
a 0.15°-wide strip at $0.5° < |\phi_1| < 2.5°$, and compare it with the published track (Shipp et al. 2019,
https://ui.adsabs.harvard.edu/abs/2019ApJ...885....3S; from the `galstreams` library, Mateu 2023). The significance of each tail is the Poisson probability of the counts in the strip
given the density in control strips at $0.6° < |\phi_2| < 1.4°$ on both sides.

In [ ]:
def rotate(x, y, pa):
    """Coordinates along (phi1) and across (phi2) a line at position angle pa (deg, east of north) through the origin."""
    t = np.radians(pa)
    return x * np.sin(t) + y * np.cos(t), x * np.cos(t) - y * np.sin(t)


# Scan the strip orientation in 1-deg steps: count masked stars within |phi2| < 0.075 deg (a 0.15-deg-wide strip) and
# 0.5 deg < |phi1| < 2.5 deg, i.e. along both tails but excluding the bound core within 0.5 deg
pas = np.arange(0, 180, 1.0)
score = []
for pa in pas:
    f1, f2 = rotate(x3[sel3], y3[sel3], pa)
    score.append(np.sum((np.abs(f2) < 0.075) & (np.abs(f1) > 0.5) & (np.abs(f1) < 2.5)))
pa_best = pas[np.argmax(score)]

# Published track (Shipp et al. 2019) from the galstreams library, at a fixed commit so that the file cannot change
GALSTREAMS = ('https://raw.githubusercontent.com/cmateu/galstreams/6a304c05bea7f774390a11082ea127719e99b70a/'
              'galstreams/tracks/track.st.TucanaIII.shipp2019.ecsv')
gs = pd.read_csv(GALSTREAMS, comment='#')
gx, gy = tangent(gs.ra.to_numpy(), gs.dec.to_numpy(), t3.ra, t3.dec)
near = np.hypot(gx, gy) < 2.5                    # the part of the track covered by the strip
# straight-line fit xi = a * eta + b to the track: its slope is tan(PA)
pa_track = np.degrees(np.arctan(np.polyfit(gy[near], gx[near], 1)[0])) % 180     # dxi/deta = tan(PA)
dpa3 = abs((pa_best - pa_track + 90) % 180 - 90)    # folded to 0-90 deg
print(f'best tail position angle {pa_best:.0f}°; galstreams (Shipp et al. 2019) track {pa_track:.1f}°; difference {dpa3:.1f}°')

# Significance of each tail: counts in the 0.15-deg strip at 0.5-2.5 deg from the core, against the density in control
# strips 0.6 < |phi2| < 1.4 deg on both sides of the stream (same phi1 range), scaled by the width ratio
f1, f2 = rotate(x3[sel3], y3[sel3], pa_best)
res = {}
for side, s in (('leading/east', 1), ('trailing/west', -1)):
    along = (s * f1 > 0.5) & (s * f1 < 2.5)
    n_on = np.sum(along & (np.abs(f2) < 0.075))
    n_ctrl = np.sum(along & (np.abs(f2) > 0.6) & (np.abs(f2) < 1.4))
    lam = n_ctrl * 0.15 / 1.6                                       # control strips are 1.6 deg wide in total
    p_ = stats.poisson.sf(n_on - 1, lam)                            # P(N >= n_on) for a Poisson mean lam
    res[side] = stats.norm.isf(p_)                                  # the same tail probability in Gaussian sigma
    print(f'{side:14s}: {n_on} stars in the strip, {lam:.1f} expected from control strips -> {res[side]:.1f} sigma')

# Left: strip counts against position angle, with the galstreams PA (dashed). Right: profile across the best strip
# orientation, for 0.5 < |phi1| < 2.5 deg, in 0.05-deg bins over +-1.5 deg (covers the control strips)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(pas, score)
ax[0].axvline(pa_track, c='C1', ls='--', label='galstreams track')
ax[0].set(xlabel='position angle of the strip [deg]', ylabel='masked stars in the strip')
ax[0].legend()
ax[1].hist(f2[(np.abs(f1) > 0.5) & (np.abs(f1) < 2.5)], np.arange(-1.5, 1.51, 0.05), histtype='step', lw=2)
ax[1].set(xlabel=r'$\phi_2$ [deg] (across the tails)', ylabel='masked stars, 0.5° < |φ₁| < 2.5°')
plt.tight_layout()
# PA within 10 deg of the published track, and both tails detected at more than 3 sigma (STRUCTURE.md, Chapter 11)
assert dpa3 < 10 and min(res.values()) > 3

**Figure 11.3.** Left: number of matched-filter stars in a 0.15°-wide strip ($|\phi_2| < 0.075°$) at
$0.5° < |\phi_1| < 2.5°$ through Tucana III, as a function of the strip's position angle (blue, 1° steps); the dashed
orange line is the position angle of the published track (Shipp et al. 2019, from `galstreams`), 85.8°. The counts peak
at 85° (≈ 820 stars, against ≈ 450–550 at other orientations), 0.8° from the track. Right: histogram of $\phi_2$, the
distance across the best strip, for the stars at $0.5° < |\phi_1| < 2.5°$, in 0.05° bins: a narrow stream (~0.1° wide,
i.e. 0.1° × π/180 × 23 kpc ≈ 40 pc) on a background that rises slowly from ≈ 140 stars per bin at $\phi_2 = -1.5°$ to
≈ 175 at +1.5°. The strip holds 386 stars on the eastern (leading) side and 435 on the western (trailing) side, against
218.6 and 264.1 expected from the control strips at $0.6° < |\phi_2| < 1.4°$: both tails are detected well above 3σ
(10.2σ and 9.6σ).

Tucana III shows the continuity between the two kinds of substructure of this tutorial: a bound core (a satellite) and
the stars it has lost (a stream), seen together.

## Summary

- Moving to a new survey means adopting its magnitudes, star–galaxy flag, extinction coefficients and isochrone system;
  the methods stay the same.
- DES depth reaches the turn-off of satellites at ~30 kpc: Reticulum II's structure is measured from hundreds of stars,
  in agreement with deeper imaging.
- Tucana III's tails are found with the matched filter and the strip test of Chapter 7, along the published track.